# Phishing URL Detection - Initial Exploration

## Business Context

**Stakeholder**: Small security consulting firm (internal IT security team)

**Problem Statement**: Staff members are increasingly targeted by phishing attacks via suspicious URLs in emails and messages. The firm needs a quick-check tool where any employee can paste a URL and get an instant risk assessment.

**Business Goal**: 
- Reduce phishing click-through incidents by 70% within 6 months
- Provide real-time URL risk scoring with explainable features
- Enable rapid security awareness training by showing WHY a URL is risky (excessive subdomains, suspicious characters, unusual length, etc.)

**Success Criteria**: A lightweight classification model that can flag risky URLs with high precision, minimizing false alarms while catching genuine threats.

## CRISP-DM Methodology

We're following the **CRISP-DM** (Cross-Industry Standard Process for Data Mining) framework:

1. **Business Understanding** - Define the phishing detection problem and success metrics
2. **Data Understanding** (this notebook) - Load, explore, and identify key patterns in URL features
3. **Data Preparation** - Feature engineering, cleaning, train/test splits
4. **Modeling** - Build and tune classification models
5. **Evaluation** - Assess performance against business KPIs
6. **Deployment** - Create a simple prediction interface

**This session**: Phases 1-2 (Business Understanding + Data Understanding)

In [ ]:
# Unpack the dataset archive
import zipfile
import os

# Extract datasets from archive
with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('data/')
    
print("Extracted files:")
for file in os.listdir('data/'):
    size_mb = os.path.getsize(f'data/{file}') / (1024 * 1024)
    print(f"  - {file} ({size_mb:.1f} MB)")

In [ ]:
# Load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully")

In [ ]:
# Load the first dataset to get a feel for the data structure
df = pd.read_csv('data/dataset1.csv')

print(f"Dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Check all datasets - test loading with proper error handling
import glob
import warnings

dataset_files = sorted(glob.glob('data/dataset*.csv'))

print("Comparing all datasets:\n")
for file in dataset_files:
    issues = []
    
    try:
        # Try UTF-8 first
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            df_temp = pd.read_csv(file, encoding='utf-8', low_memory=False)
            if w:
                for warning in w:
                    if 'DtypeWarning' in str(warning.category):
                        issues.append("mixed dtypes")
    except UnicodeDecodeError:
        # Try latin-1 encoding
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                df_temp = pd.read_csv(file, encoding='latin-1', on_bad_lines='skip', low_memory=False)
                issues.append("encoding: latin-1")
        except Exception as e:
            print(f"{file}")
            print(f"  FAILED: {str(e)[:80]}")
            print()
            continue
    except Exception as e:
        # Handle parsing errors
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                df_temp = pd.read_csv(file, encoding='latin-1', on_bad_lines='skip', low_memory=False)
                issues.append("parsing errors")
        except Exception as e2:
            print(f"{file}")
            print(f"  FAILED: {str(e2)[:80]}")
            print()
            continue
    
    print(f"{file}")
    print(f"  Shape: {df_temp.shape[0]:,} rows × {df_temp.shape[1]} columns")
    print(f"  Columns: {df_temp.columns.tolist()[:5]}...")
    if issues:
        print(f"  Issues: {', '.join(issues)}")
    print()

## Key Finding: Heterogeneous Dataset Collection

**Discovery**: While the Kaggle page mentions "diverse approaches," loading the data reveals just how different these datasets are. This is the first time we're seeing the actual structure directly.

**What We Found**:
- 6 datasets with wildly different structures (32 to 112 columns)
- Sizes range from 10k to 235k rows
- Different feature engineering philosophies (some extract 100+ features, others focus on 14 core signals)
- dataset5 has data quality issues (parsing errors, mixed data types)

**Implication for Our Business Case**:

Our stakeholder needs **explainability** - when a URL is flagged, staff need to understand WHY. This means we need human-readable feature names, not just numerical indicators.

**Selection Criteria**:
1. Contains raw URL or domain column (for showing examples)
2. Feature names are interpretable (e.g., "NumDots" not "feature_47")
3. Sufficient training data (10k+ minimum)
4. Clean loading (no major data quality issues)

**Top Candidates**:
- **dataset3**: 11k rows, 89 features - has 'url', 'length_url', 'nb_dots' (interpretable)
- **dataset4**: 235k rows, 56 features - has 'URL', 'Domain', 'URLLength' (LARGEST, interpretable)
- **dataset6**: 10k rows, 50 features - has 'NumDots', 'SubdomainLevel' (interpretable but smallest)

**Decision**: Examine dataset3 and dataset4 in detail. dataset4's size (235k) is compelling for model performance, but we need to verify feature interpretability first.

In [ ]:
# Examine dataset4 in detail (largest candidate at 235k rows)
df4 = pd.read_csv('data/dataset4.csv')

print(f"=== Dataset4 Overview ===")
print(f"Shape: {df4.shape[0]:,} rows × {df4.shape[1]} columns\n")

# Create a DataFrame to display column names in a clean format
col_info = pd.DataFrame({
    'Column': df4.columns,
    'Type': df4.dtypes.values
})

print("Features:")
print(col_info.to_string(index=False))

In [ ]:
# Sample rows with key features to understand what the data captures
# Select a diverse subset of features that tell different parts of the story

key_features = [
    'URL',                    # What we're analyzing
    'Domain',                 # Domain extracted
    'URLLength',              # Structure: length
    'NoOfSubDomain',          # Structure: subdomain count
    'IsHTTPS',                # Security: HTTPS flag
    'IsDomainIP',             # Security: IP address instead of domain
    'Bank',                   # Content: banking keyword
    'Pay',                    # Content: payment keyword  
    'Crypto',                 # Content: crypto keyword
    'HasPasswordField',       # Behavior: has password input
    'HasObfuscation',         # Behavior: obfuscated characters
    'label'                   # Ground truth (0=legit, 1=phishing)
]

# Show 10 samples to get diverse examples
sample_df = df4[key_features].head(10)

# Set pandas display options for better readability
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', None)

print("=== Sample URLs with Key Features ===\n")
print(sample_df.to_string(index=False))

# Reset display options
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')

In [ ]:
# Verify what the label values actually mean
# Check class distribution first
print("=== Label Distribution ===")
print(df4['label'].value_counts().sort_index())
print(f"\nClass balance:")
print(df4['label'].value_counts(normalize=True).sort_index())

print("\n=== Examples of label=0 ===")
label_0_sample = df4[df4['label'] == 0][['URL', 'Domain', 'IsHTTPS', 'HasObfuscation', 'Bank', 'label']].head(5)
print(label_0_sample.to_string(index=False))

print("\n=== Examples of label=1 ===")
label_1_sample = df4[df4['label'] == 1][['URL', 'Domain', 'IsHTTPS', 'HasObfuscation', 'Bank', 'label']].head(5)
print(label_1_sample.to_string(index=False))

## Correction: Label Encoding

**Previous Assumption (INCORRECT)**: In cell-8, we assumed `label=0` meant legitimate and `label=1` meant phishing.

**Evidence from verification**:
- **label=0** examples: `f0519141.xsph.ru`, `shprakserf.gq`, `kuradox92.lima-city.de` - suspicious domains with random strings, uncommon TLDs
- **label=1** examples: `uni-mainz.de`, `voicefmradio.co.uk`, `rewildingargentina.org` - recognizable legitimate organizations

**Actual encoding**:
- **0 = phishing** (42.8% of dataset, 100,945 samples)
- **1 = legitimate** (57.2% of dataset, 134,850 samples)

**Lesson**: Always verify assumptions about data encoding before proceeding with analysis. The comment in cell-8 remains as a reminder of this error.

# Dataset4 Feature Deep Dive

Now that we've selected dataset4 (235k rows, 56 features) as our primary dataset, we need to understand what each feature actually measures. This understanding is critical for:

1. **Rule-based system** - Identifying clear red flags for instant detection
2. **ML model** - Understanding feature importance and model decisions  
3. **Explainability** - Explaining to users WHY a URL was flagged

We'll examine non-obvious features one by one, looking at:
- What the feature captures
- Value distributions (phishing vs legitimate)
- Real URL examples demonstrating the feature